# Lab 2: Fine-Tuning Workflow

In this lab, we implement a complete multi-phase fine-tuning workflow. Starting from feature extraction (frozen base), we progressively unfreeze layers and fine-tune with a reduced learning rate, using callbacks to prevent overfitting.

## Learning Objectives
- Implement a multi-phase fine-tuning workflow
- Use reduced learning rates for fine-tuning pre-trained layers
- Apply EarlyStopping and ReduceLROnPlateau callbacks
- Compare learning curves across training phases
- Experiment with unfreezing different numbers of layers

In [ ]:
# Run this cell in Google Colab to install dependencies
# Skip if running locally with uv
import sys
if 'google.colab' in sys.modules:
    !pip install -q keras torch torchvision gradio python-dotenv datasets transformers huggingface_hub
    print('Dependencies installed!')

## 1. Setup and Installation

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"

import keras
import numpy as np
import matplotlib.pyplot as plt
import torch

print(f"Keras version: {keras.__version__}")
print(f"Keras backend: {keras.backend.backend()}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Load and Prepare the Dataset

Same binary classification task as Lab 1: airplane vs automobile from CIFAR-10.

In [ ]:
# Load CIFAR-10 and filter for binary classification
(x_train_full, y_train_full), (x_test_full, y_test_full) = keras.datasets.cifar10.load_data()

# Filter for airplane (0) and automobile (1)
train_mask = (y_train_full.flatten() == 0) | (y_train_full.flatten() == 1)
test_mask = (y_test_full.flatten() == 0) | (y_test_full.flatten() == 1)

x_train = x_train_full[train_mask].astype("float32") / 255.0
y_train = y_train_full[train_mask].flatten()
x_test = x_test_full[test_mask].astype("float32") / 255.0
y_test = y_test_full[test_mask].flatten()

class_names = ["airplane", "automobile"]

print(f"Training set: {x_train.shape}")
print(f"Test set: {x_test.shape}")

## 3. Phase 1: Feature Extraction (Frozen Base)

First, we train only the classification head with the base model completely frozen.

In [ ]:
# Load pre-trained MobileNetV2
base_model = keras.applications.MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(96, 96, 3)
)

# Freeze all layers in the base model
base_model.trainable = False

# Build the model
inputs = keras.Input(shape=(32, 32, 3))
x = keras.layers.Resizing(96, 96)(inputs)
x = keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dropout(0.2)(x)
outputs = keras.layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, outputs, name="fine_tuning_model")

print(f"Total layers in base model: {len(base_model.layers)}")
print(f"Trainable weights: {len(model.trainable_weights)}")

In [ ]:
# Compile for Phase 1
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Train Phase 1: feature extraction
print("=" * 50)
print("Phase 1: Feature Extraction (frozen base)")
print("=" * 50)

history_phase1 = model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

phase1_loss, phase1_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"\nPhase 1 Test Accuracy: {phase1_acc:.4f}")

## 4. Phase 2: Fine-Tuning Top Layers

Now we unfreeze the top layers of the base model and continue training with a much lower learning rate. This allows the top layers to adapt to our specific task while preserving the general features learned from ImageNet.

In [ ]:
# Unfreeze the base model
base_model.trainable = True

# Freeze all layers except the last 20
num_layers_to_unfreeze = 20
for layer in base_model.layers[:-num_layers_to_unfreeze]:
    layer.trainable = False

# Count trainable layers
trainable_layers = sum(1 for layer in base_model.layers if layer.trainable)
frozen_layers = sum(1 for layer in base_model.layers if not layer.trainable)
print(f"Base model trainable layers: {trainable_layers}")
print(f"Base model frozen layers: {frozen_layers}")
print(f"Total trainable weights: {len(model.trainable_weights)}")

In [ ]:
# Recompile with a much lower learning rate
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Train Phase 2: fine-tuning
print("=" * 50)
print("Phase 2: Fine-Tuning (top layers unfrozen, lr=1e-5)")
print("=" * 50)

history_phase2 = model.fit(
    x_train, y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

phase2_loss, phase2_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"\nPhase 2 Test Accuracy: {phase2_acc:.4f}")
print(f"Improvement over Phase 1: {(phase2_acc - phase1_acc) * 100:.2f} percentage points")

## 5. Phase 3: Fine-Tuning with Callbacks

We continue training with EarlyStopping and ReduceLROnPlateau callbacks to find the optimal stopping point and learning rate.

In [ ]:
# Define callbacks
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-7,
    verbose=1
)

# Recompile (reset optimizer state)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Train Phase 3: fine-tuning with callbacks
print("=" * 50)
print("Phase 3: Fine-Tuning with Callbacks")
print("=" * 50)

history_phase3 = model.fit(
    x_train, y_train,
    epochs=15,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

phase3_loss, phase3_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"\nPhase 3 Test Accuracy: {phase3_acc:.4f}")

## 6. Plot Learning Curves Across All Phases

In [ ]:
# Combine histories for plotting
def combine_histories(*histories):
    combined = {}
    for h in histories:
        for key, values in h.history.items():
            if key not in combined:
                combined[key] = []
            combined[key].extend(values)
    return combined

# Plot Phase 1 and Phase 2 separately
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Phase 1 - Accuracy
axes[0, 0].plot(history_phase1.history["accuracy"], label="Train", marker="o")
axes[0, 0].plot(history_phase1.history["val_accuracy"], label="Validation", marker="s")
axes[0, 0].set_title("Phase 1: Feature Extraction - Accuracy")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Accuracy")
axes[0, 0].legend()
axes[0, 0].grid(True)

# Phase 1 - Loss
axes[0, 1].plot(history_phase1.history["loss"], label="Train", marker="o")
axes[0, 1].plot(history_phase1.history["val_loss"], label="Validation", marker="s")
axes[0, 1].set_title("Phase 1: Feature Extraction - Loss")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("Loss")
axes[0, 1].legend()
axes[0, 1].grid(True)

# Phase 2 - Accuracy
axes[1, 0].plot(history_phase2.history["accuracy"], label="Train", marker="o")
axes[1, 0].plot(history_phase2.history["val_accuracy"], label="Validation", marker="s")
axes[1, 0].set_title("Phase 2: Fine-Tuning - Accuracy")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("Accuracy")
axes[1, 0].legend()
axes[1, 0].grid(True)

# Phase 2 - Loss
axes[1, 1].plot(history_phase2.history["loss"], label="Train", marker="o")
axes[1, 1].plot(history_phase2.history["val_loss"], label="Validation", marker="s")
axes[1, 1].set_title("Phase 2: Fine-Tuning - Loss")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("Loss")
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Combined view across all phases
all_acc = (history_phase1.history["val_accuracy"] +
           history_phase2.history["val_accuracy"] +
           history_phase3.history["val_accuracy"])
all_loss = (history_phase1.history["val_loss"] +
            history_phase2.history["val_loss"] +
            history_phase3.history["val_loss"])

phase1_end = len(history_phase1.history["val_accuracy"])
phase2_end = phase1_end + len(history_phase2.history["val_accuracy"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(all_acc, marker="o", markersize=4)
ax1.axvline(x=phase1_end - 0.5, color="red", linestyle="--", label="Phase 1 -> 2")
ax1.axvline(x=phase2_end - 0.5, color="orange", linestyle="--", label="Phase 2 -> 3")
ax1.set_title("Validation Accuracy Across All Phases")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy")
ax1.legend()
ax1.grid(True)

ax2.plot(all_loss, marker="o", markersize=4)
ax2.axvline(x=phase1_end - 0.5, color="red", linestyle="--", label="Phase 1 -> 2")
ax2.axvline(x=phase2_end - 0.5, color="orange", linestyle="--", label="Phase 2 -> 3")
ax2.set_title("Validation Loss Across All Phases")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## 7. Experiment: Unfreezing Different Numbers of Layers

Let's see how the number of unfrozen layers affects performance.

In [ ]:
def build_and_train(num_unfreeze, epochs_phase1=3, epochs_phase2=5):
    """Build and train a model with a specified number of unfrozen layers."""
    # Build model
    base = keras.applications.MobileNetV2(
        weights="imagenet", include_top=False, input_shape=(96, 96, 3)
    )
    base.trainable = False

    inp = keras.Input(shape=(32, 32, 3))
    x = keras.layers.Resizing(96, 96)(inp)
    x = keras.applications.mobilenet_v2.preprocess_input(x)
    x = base(x, training=False)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(0.2)(x)
    out = keras.layers.Dense(1, activation="sigmoid")(x)
    m = keras.Model(inp, out)

    # Phase 1: feature extraction
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=["accuracy"])
    m.fit(x_train, y_train, epochs=epochs_phase1, batch_size=64, validation_split=0.2, verbose=0)

    # Phase 2: fine-tune
    if num_unfreeze > 0:
        base.trainable = True
        for layer in base.layers[:-num_unfreeze]:
            layer.trainable = False
        m.compile(optimizer=keras.optimizers.Adam(1e-5), loss="binary_crossentropy", metrics=["accuracy"])
        m.fit(x_train, y_train, epochs=epochs_phase2, batch_size=64, validation_split=0.2, verbose=0)

    _, acc = m.evaluate(x_test, y_test, verbose=0)
    return acc

# Test different numbers of unfrozen layers
unfreeze_options = [0, 10, 20, 40, 80]
results = {}

for n in unfreeze_options:
    print(f"Testing with {n} unfrozen layers...", end=" ")
    acc = build_and_train(n)
    results[n] = acc
    print(f"Accuracy: {acc:.4f}")

print("\n--- Summary ---")
for n, acc in results.items():
    label = "(frozen)" if n == 0 else f"(unfreeze {n})"
    print(f"  {label}: {acc:.4f}")

In [ ]:
# Plot the experiment results
plt.figure(figsize=(8, 5))
plt.bar([str(n) for n in results.keys()], results.values(), color="steelblue")
plt.xlabel("Number of Unfrozen Layers")
plt.ylabel("Test Accuracy")
plt.title("Effect of Unfreezing Different Numbers of Layers")
plt.ylim(0.85, 1.0)
for i, (n, acc) in enumerate(results.items()):
    plt.text(i, acc + 0.002, f"{acc:.4f}", ha="center", fontsize=10)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Summary

In [ ]:
print("=" * 50)
print("Fine-Tuning Workflow Summary")
print("=" * 50)
print(f"Phase 1 (Feature Extraction): {phase1_acc:.4f}")
print(f"Phase 2 (Fine-Tuning, 20 layers): {phase2_acc:.4f}")
print(f"Phase 3 (Fine-Tuning + Callbacks): {phase3_acc:.4f}")
print(f"\nKey Takeaways:")
print(f"  1. Feature extraction provides a strong baseline quickly")
print(f"  2. Fine-tuning top layers can further improve accuracy")
print(f"  3. Using callbacks prevents overfitting and finds optimal stopping")
print(f"  4. A low learning rate (1e-5) is crucial for fine-tuning")

## 9. Gradio Interface

Interactive classification with the fine-tuned model.

In [ ]:
import gradio as gr

def classify_image(image):
    """Classify an uploaded image as airplane or automobile."""
    if image is None:
        return {"airplane": 0.0, "automobile": 0.0}

    # Preprocess
    img = np.array(image)
    if img.ndim == 2:
        img = np.stack([img] * 3, axis=-1)
    if img.shape[-1] == 4:
        img = img[:, :, :3]

    from PIL import Image
    img_pil = Image.fromarray(img.astype(np.uint8)).resize((32, 32))
    img_array = np.array(img_pil).astype("float32") / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array, verbose=0)[0][0]

    return {
        "airplane": float(1 - prediction),
        "automobile": float(prediction)
    }

demo = gr.Interface(
    fn=classify_image,
    inputs=gr.Image(type="numpy", label="Upload an Image"),
    outputs=gr.Label(num_top_classes=2, label="Classification"),
    title="Fine-Tuned Classifier",
    description="Upload an image to classify it as an airplane or automobile using a fine-tuned MobileNetV2 model.",
    examples=None,
    flagging_mode="never"
)

demo.launch(share=False)